# ComfyUI Colab (r2)

r1からの主な改善点:
- ✅ `simpleeval` / `pyopengl` 追加 → nodes_math / nodes_glsl エラー解消
- ✅ torch 再インストール行を削除 → Colab既存の cu128 版をそのまま利用
- ✅ `comfy_kitchen` 追加 → fp8/fp4 量子化対応
- ✅ Google Drive マウントを `USE_GOOGLE_DRIVE` フラグと統合
- ✅ モデルダウンロードを `huggingface_hub` に変更 → リトライ・整合性チェック付き
- ✅ xvfb 仮想ディスプレイ設定 → GLSLノード有効化

## Step 1: Environment Setup
ComfyUI のセットアップ、依存パッケージのインストールを行います。

In [ ]:
#@title Environment Setup

from pathlib import Path

OPTIONS = {}

USE_GOOGLE_DRIVE = False  #@param {type:"boolean"} Falseにすると100GBエリアへのインストールに切り替わります
UPDATE_COMFY_UI = True   #@param {type:"boolean"}
USE_COMFYUI_MANAGER = True  #@param {type:"boolean"}
INSTALL_CUSTOM_NODES_DEPENDENCIES = True  #@param {type:"boolean"}

OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI
OPTIONS['USE_COMFYUI_MANAGER'] = USE_COMFYUI_MANAGER
OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES'] = INSTALL_CUSTOM_NODES_DEPENDENCIES

current_dir = !pwd
WORKSPACE = f"{current_dir[0]}/ComfyUI"

# ── Google Drive 連動（USE_GOOGLE_DRIVE フラグで制御）──
if OPTIONS['USE_GOOGLE_DRIVE']:
    !echo "Mounting Google Drive..."
    %cd /
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
    %cd /content/drive/MyDrive
    print(f"✅ Google Drive モード: 画像は {WORKSPACE}/output に保存されます")
else:
    print(f"✅ ローカルモード: 画像は {WORKSPACE}/output に保存されます")

# ── ComfyUI のクローン / 更新 ──
![ ! -d $WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
    !echo -= Updating ComfyUI =-
    ![ -f ".ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat
    ![ -f ".ci/nightly/windows_base_files/run_nvidia_gpu.bat" ] && chmod 755 .ci/nightly/windows_base_files/run_nvidia_gpu.bat
    ![ -f ".ci/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows/update_comfyui_and_python_dependencies.bat
    ![ -f ".ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat
    ![ -f ".ci/update_windows/update.py" ] && chmod 755 .ci/update_windows/update.py
    ![ -f ".ci/update_windows/update_comfyui.bat" ] && chmod 755 .ci/update_windows/update_comfyui.bat
    ![ -f ".ci/update_windows/README_VERY_IMPORTANT.txt" ] && chmod 755 .ci/update_windows/README_VERY_IMPORTANT.txt
    ![ -f ".ci/update_windows/run_cpu.bat" ] && chmod 755 .ci/update_windows/run_cpu.bat
    ![ -f ".ci/update_windows/run_nvidia_gpu.bat" ] && chmod 755 .ci/update_windows/run_nvidia_gpu.bat
    !git pull

# ── 依存パッケージ ──
!echo -= Install dependencies =-
!pip install -q accelerate
!pip install -q einops transformers>=4.28.1 safetensors>=0.4.2 aiohttp pyyaml Pillow scipy tqdm psutil tokenizers>=0.13.3

# 【r2改善①】torch は Colab 既存の cu128 版をそのまま使う（再インストール不要）
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121  ← 削除
print("ℹ️  torch: Colab 既存ビルドを使用 (再インストールをスキップ)")
import torch; print(f"   torch version: {torch.__version__}")

!pip install -q torchsde
!pip install -q kornia>=0.7.1 spandrel soundfile sentencepiece
!pip install -q comfyui-workflow-templates
!pip install -q comfyui-embedded-docs

# 【r2改善②】nodes_math / nodes_glsl エラーを解消する追加パッケージ
!pip install -q simpleeval
!pip install -q pyopengl

# 【r2改善③】fp8/fp4 量子化対応
!pip install -q comfy-kitchen

# ── ComfyUI-Manager ──
if OPTIONS['USE_COMFYUI_MANAGER']:
    %cd custom_nodes
    ![ -f "ComfyUI-Manager/check.sh" ] && chmod 755 ComfyUI-Manager/check.sh
    ![ -f "ComfyUI-Manager/scan.sh" ] && chmod 755 ComfyUI-Manager/scan.sh
    ![ -f "ComfyUI-Manager/node_db/dev/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/dev/scan.sh
    ![ -f "ComfyUI-Manager/node_db/tutorial/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/tutorial/scan.sh
    ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh
    ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-win.bat" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-win.bat
    ![ ! -d ComfyUI-Manager ] && echo -= Initial setup ComfyUI-Manager =- && git clone https://github.com/ltdrdata/ComfyUI-Manager
    %cd ComfyUI-Manager
    !git pull

# ComfyUI-Impact-Pack
![ ! -d $WORKSPACE/custom_nodes/ComfyUI-Impact-Pack ] && git clone https://github.com/ltdrdata/ComfyUI-Impact-Pack.git $WORKSPACE/custom_nodes/ComfyUI-Impact-Pack

# ComfyUI_IPAdapter_plus（キャラクター参照画像を使ったimg2imgに使用）
![ ! -d $WORKSPACE/custom_nodes/ComfyUI_IPAdapter_plus ] && git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus.git $WORKSPACE/custom_nodes/ComfyUI_IPAdapter_plus

%cd $WORKSPACE

if OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES']:
    !echo -= Install custom nodes dependencies =-
    !pip install -q GitPython
    !python custom_nodes/ComfyUI-Manager/cm-cli.py restore-dependencies

# rembg / onnxruntime / insightface
!pip install -q rembg onnxruntime insightface

# av / comfy_aimdo
!pip install -q av comfy_aimdo

print("\n✅ Step 1 完了")

## Step 2: 仮想ディスプレイ設定（GLSLノード有効化）
Colab はヘッドレス環境のため、OpenGL を必要とする `nodes_glsl.py` を使うには xvfb が必要です。

In [2]:
#@title 【r2改善④】xvfb 仮想ディスプレイのセットアップ（GLSLノード有効化）

!apt-get install -y -q xvfb
import subprocess, os, time

# 既存の :99 プロセスがあれば終了
subprocess.run(["pkill", "-f", "Xvfb :99"], capture_output=True)
time.sleep(0.5)

# 仮想ディスプレイ起動
subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1024x768x24"])
os.environ['DISPLAY'] = ':99'
time.sleep(1)

print("✅ 仮想ディスプレイ :99 を起動しました (DISPLAY=:99)")

Reading package lists...
Building dependency tree...
Reading state information...
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.16).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
✅ 仮想ディスプレイ :99 を起動しました (DISPLAY=:99)


## Step 3: 出力フォルダの設定
生成画像を常に Google Drive の  へ保存します。


In [3]:
#@title 【r2改善⑤】出力先設定（常に Google Drive へ保存）

import os, shutil
from google.colab import drive

# Google Drive マウント
drive.mount("/content/drive", force_remount=False)

# WORKSPACE を再定義（Step1 未実行でも動くよう独立させる）
_workspace = "/content/ComfyUI"

# Drive 側の出力フォルダを作成
drive_output = "/content/drive/MyDrive/ComfyUI_Output"
os.makedirs(drive_output, exist_ok=True)

# ComfyUI の output をシンボリックリンクで差し替え
local_output = f"{_workspace}/output"
if os.path.islink(local_output):
    os.unlink(local_output)
elif os.path.isdir(local_output):
    shutil.rmtree(local_output)

os.symlink(drive_output, local_output)

# 確認
assert os.path.islink(local_output)
assert os.path.exists(local_output)
print("output ->", os.path.realpath(local_output))
print("setup complete: images will be saved to MyDrive/ComfyUI_Output")


Mounted at /content/drive
output -> /content/drive/MyDrive/ComfyUI_Output
setup complete: images will be saved to MyDrive/ComfyUI_Output


## Step 4: モデルのダウンロード
`huggingface_hub` を使ってリトライ付き・整合性チェック付きでダウンロードします。

| モデルファイル | 用途 | デフォルト |
|---|---|---|
| `anima-base-v1.0.safetensors` | diffusion model（推奨） | ✅ ON |
| `anima-preview.safetensors` | diffusion model（旧プレビュー版） | OFF |
| `qwen_3_06b_base.safetensors` | text encoder（共通） | 常時 |
| `qwen_image_vae.safetensors` | VAE（共通） | 常時 |

In [ ]:
#@title 【r2改善⑥】Anima モデルのダウンロード（base v1.0 対応版）

from huggingface_hub import hf_hub_download
import os, shutil, glob

REPO_ID = "circlestone-labs/Anima"
MODEL_BASE = f"{WORKSPACE}/models"

# ダウンロードするモデルを選択
DOWNLOAD_BASE_V1  = True   #@param {type:"boolean"} base v1.0（推奨）
DOWNLOAD_PREVIEW  = False  #@param {type:"boolean"} preview（旧バージョン）

def download_model(repo_id, hf_filename, local_dir):
    """
    hf_hub_download は filename のサブディレクトリ構造をそのまま再現するため、
    ダウンロード後に目的のフォルダへ移動する。
    """
    os.makedirs(local_dir, exist_ok=True)
    basename = os.path.basename(hf_filename)
    dest = os.path.join(local_dir, basename)
    if os.path.exists(dest):
        print(f"skip (exists): {basename}")
        return
    print(f"downloading: {basename} ...")
    tmp_dir = f"{WORKSPACE}/_hf_tmp"
    downloaded_path = hf_hub_download(
        repo_id=repo_id,
        filename=hf_filename,
        local_dir=tmp_dir,
    )
    shutil.move(downloaded_path, dest)
    shutil.rmtree(tmp_dir, ignore_errors=True)
    print(f"done: {basename} -> {dest}")

# ── diffusion model ──
if DOWNLOAD_BASE_V1:
    download_model(REPO_ID,
        "split_files/diffusion_models/anima-base-v1.0.safetensors",
        f"{MODEL_BASE}/diffusion_models")

if DOWNLOAD_PREVIEW:
    download_model(REPO_ID,
        "split_files/diffusion_models/anima-preview.safetensors",
        f"{MODEL_BASE}/diffusion_models")

# ── 共通モデル（base v1.0 / preview 共通）──
download_model(REPO_ID,
    "split_files/text_encoders/qwen_3_06b_base.safetensors",
    f"{MODEL_BASE}/text_encoders")

download_model(REPO_ID,
    "split_files/vae/qwen_image_vae.safetensors",
    f"{MODEL_BASE}/vae")

# 保存先を確認
print("")
print("saved model files:")
for p in sorted(glob.glob(f"{MODEL_BASE}/**/*.safetensors", recursive=True)):
    print(" ", p)

print("")
print("model download complete")
print("")
print("【base v1.0 推奨パラメータ】")
print("  解像度: 512² ～ 1536²")
print("  Steps : 30-50  /  CFG: 4-5")
print("  Sampler: er_sde（デフォルト）/ euler_a / dpmpp_2m_sde_gpu")

In [5]:
#@title オプション: その他のモデル（コメントアウトを外して使用）

# from huggingface_hub import hf_hub_download
# MODEL_BASE = f"{WORKSPACE}/models"

# ── SDXL ──
# hf_hub_download("stabilityai/stable-diffusion-xl-base-1.0",
#     "sd_xl_base_1.0.safetensors", local_dir=f"{MODEL_BASE}/checkpoints", local_dir_use_symlinks=False)
# hf_hub_download("stabilityai/stable-diffusion-xl-refiner-1.0",
#     "sd_xl_refiner_1.0.safetensors", local_dir=f"{MODEL_BASE}/checkpoints", local_dir_use_symlinks=False)

# ── FLUX.1 ──
# hf_hub_download("black-forest-labs/FLUX.1-schnell",
#     "flux1-schnell.safetensors", local_dir=f"{MODEL_BASE}/diffusion_models", local_dir_use_symlinks=False)

# ── VAE (汎用) ──
# hf_hub_download("stabilityai/sd-vae-ft-mse-original",
#     "vae-ft-mse-840000-ema-pruned.safetensors", local_dir=f"{MODEL_BASE}/vae", local_dir_use_symlinks=False)

# ── UpScale ──
# import urllib.request
# urllib.request.urlretrieve(
#     "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
#     f"{MODEL_BASE}/upscale_models/RealESRGAN_x4plus.pth")

print("オプションモデルセルです。必要なものをコメントアウト解除してください。")

オプションモデルセルです。必要なものをコメントアウト解除してください。


In [ ]:
#@title Step 4b: IP-Adapter モデルのダウンロード（ComfyUI_IPAdapter_plus 用）

from huggingface_hub import hf_hub_download
import os, shutil, glob

MODEL_BASE      = f"{WORKSPACE}/models"
CLIP_VISION_DIR = f"{MODEL_BASE}/clip_vision"
IPADAPTER_DIR   = f"{MODEL_BASE}/ipadapter"

# ── ダウンロードするモデルを選択 ──
DOWNLOAD_CLIP_VIT_H    = True   #@param {type:"boolean"} CLIP ViT-H-14（Plus系モデルで必須）
DOWNLOAD_IPA_PLUS_SD15 = True   #@param {type:"boolean"} ip-adapter-plus_sd15（SD1.5用）
DOWNLOAD_IPA_PLUS_SDXL = False  #@param {type:"boolean"} ip-adapter-plus_sdxl_vit-h（SDXL用）

def download_ipa(repo_id, hf_filename, local_dir, save_as=None):
    os.makedirs(local_dir, exist_ok=True)
    basename = save_as if save_as else os.path.basename(hf_filename)
    dest = os.path.join(local_dir, basename)
    if os.path.exists(dest):
        print(f"skip (exists): {basename}")
        return
    print(f"downloading: {basename} ...")
    tmp_dir = f"{WORKSPACE}/_hf_tmp_ipa"
    downloaded_path = hf_hub_download(
        repo_id=repo_id,
        filename=hf_filename,
        local_dir=tmp_dir,
    )
    shutil.move(downloaded_path, dest)
    shutil.rmtree(tmp_dir, ignore_errors=True)
    print(f"done: {basename} -> {dest}")

# ── CLIP Vision ViT-H-14（画像エンコーダ、Plus 系モデルで必須）──
if DOWNLOAD_CLIP_VIT_H:
    download_ipa(
        "h94/IP-Adapter",
        "models/image_encoder/model.safetensors",
        CLIP_VISION_DIR,
        save_as="CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors",
    )

# ── IP-Adapter Plus SD1.5 ──
if DOWNLOAD_IPA_PLUS_SD15:
    download_ipa(
        "h94/IP-Adapter",
        "models/ip-adapter-plus_sd15.safetensors",
        IPADAPTER_DIR,
    )

# ── IP-Adapter Plus SDXL ViT-H ──
if DOWNLOAD_IPA_PLUS_SDXL:
    download_ipa(
        "h94/IP-Adapter",
        "sdxl_models/ip-adapter-plus_sdxl_vit-h.safetensors",
        IPADAPTER_DIR,
    )

print("\n保存済みファイル:")
for p in sorted(glob.glob(f"{CLIP_VISION_DIR}/*.safetensors") +
                glob.glob(f"{IPADAPTER_DIR}/*.safetensors")):
    print(f"  {p}")

print("\n✅ IP-Adapter モデルダウンロード完了")
print("")
print("【ComfyUI ワークフローでの設定】")
print("  CLIPVisionLoader     : CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors")
print("  IPAdapterModelLoader : ip-adapter-plus_sd15.safetensors")
print("")
print("⚠️  注意: Anima (Wan 系アーキテクチャ) への IPAdapter 適用は実験的です。")
print("   上記モデルは SD1.5 / SDXL 向けのため、Anima では動作しない場合があります。")
print("   Anima 専用の IPAdapter 重みがリリースされた場合は models/ipadapter/ に追加してください。")

In [ ]:
#@title Step 4c: Anima-Turbo LoRA のダウンロード（高速生成用）

from huggingface_hub import hf_hub_download
import os, shutil, glob

MODEL_BASE = f"{WORKSPACE}/models"
LORA_DIR   = f"{MODEL_BASE}/loras"

REPO_ID = "circlestone-labs/Anima-Official-LoRAs"

# ── ダウンロードするバージョンを選択 ──
DOWNLOAD_TURBO_V02 = True   #@param {type:"boolean"} anima-turbo-lora-v0.2（最新・推奨）
DOWNLOAD_TURBO_V01 = False  #@param {type:"boolean"} anima-turbo-lora-v0.1（旧バージョン）

def download_lora(repo_id, hf_filename, local_dir):
    os.makedirs(local_dir, exist_ok=True)
    basename = os.path.basename(hf_filename)
    dest = os.path.join(local_dir, basename)
    if os.path.exists(dest):
        print(f"skip (exists): {basename}")
        return
    print(f"downloading: {basename} ...")
    tmp_dir = f"{WORKSPACE}/_hf_tmp_lora"
    downloaded_path = hf_hub_download(
        repo_id=repo_id,
        filename=hf_filename,
        local_dir=tmp_dir,
    )
    shutil.move(downloaded_path, dest)
    shutil.rmtree(tmp_dir, ignore_errors=True)
    print(f"done: {basename} -> {dest}")

if DOWNLOAD_TURBO_V02:
    download_lora(REPO_ID, "anima-turbo-lora-v0.2.safetensors", LORA_DIR)

if DOWNLOAD_TURBO_V01:
    download_lora(REPO_ID, "anima-turbo-lora-v0.1.safetensors", LORA_DIR)

print("
保存済み LoRA ファイル:")
for p in sorted(glob.glob(f"{LORA_DIR}/*.safetensors")):
    print(f"  {p}")

print("
✅ Anima-Turbo LoRA ダウンロード完了")
print("")
print("【ComfyUI ワークフローでの設定】")
print("  Load LoRA (LoraLoaderModelOnly か LoraLoader) を Load Diffusion Model の後段に接続")
print("    lora_name : anima-turbo-lora-v0.2.safetensors")
print("    strength  : 0.7〜1.0（1.0で崩れる場合は下げる）")
print("")
print("【推奨サンプリング設定（Turbo LoRA使用時）】")
print("  CFG     : 1")
print("  Steps   : 8〜12")
print("  Sampler : euler（plain）※ er_sde はノイズが乗りやすいため非推奨")
print("")
print("⚠️  Turbo LoRA は速度と引き換えに、作家性・ディテールの再現度がベースモデルよりやや落ちます。")
print("   結果が崩れる場合は strength を下げる／プロンプトの品質タグを外す／「anime coloring」を追加、などを試してください。")

In [ ]:
#@title モデル名の互換シンボリックリンク作成（旧ワークフロー対応）
# ワークフロー内で古いモデル名を参照している場合に実行してください。
# 新しいファイルへのエイリアスを作成します（実ファイルはコピーしません）。

import os

DIFFUSION_DIR = f"{WORKSPACE}/models/diffusion_models"

# 旧名 → 新名 のマッピング
ALIASES = {
    "anima-preview.safetensors": "anima-base-v1.0.safetensors",
}

created = []
for alias, real in ALIASES.items():
    alias_path = os.path.join(DIFFUSION_DIR, alias)
    real_path  = os.path.join(DIFFUSION_DIR, real)

    if os.path.exists(alias_path):
        print(f"skip (exists): {alias}")
        continue
    if not os.path.exists(real_path):
        print(f"⚠️  実ファイルが見つかりません: {real}  (Step 4 でダウンロードしてください)")
        continue

    os.symlink(real_path, alias_path)
    created.append(f"  {alias}  →  {real}")

if created:
    print("✅ シンボリックリンクを作成しました:")
    for c in created:
        print(c)
else:
    print("ℹ️  新規リンクなし（すべて処理済みまたはスキップ）")

## Step 4d: キャラクターLoRAの学習（オプション・自分の画像から）

自分で用意したキャラクター参照画像から LoRA を学習し、生成時に読み込むことでキャラクターの同一性を保ちやすくします。
学習には [kohya-ss/sd-scripts](https://github.com/kohya-ss/sd-scripts) の公式 Anima 対応(`anima_train_network.py`)を使用します。

### 事前準備
- Step 3 で Google Drive をマウント済みであること
- Google Drive 上にキャラクターの参照画像フォルダを用意（**15〜30枚程度**、正面・横顔・バストアップなどバリエーションを混ぜる）
- 可能であれば画像と同名の `.txt` キャプションファイルも用意（無ければトリガーワードのみのキャプションを自動生成します）

### 注意
- 学習は GPU を専有します。**ComfyUI を起動する前**にこのセクションを実行してください（学習後、通常どおり Step 5 で ComfyUI を起動）
- 無料版 Colab の T4 GPU では 1000 steps で 4 時間以上かかることがあります。まずは少ない epoch 数で試すことを推奨します
- sd-scripts のセットアップで一部パッケージのバージョンが変わります。学習後に ComfyUI の起動でエラーが出た場合はランタイムを再起動してください

In [ ]:
#@title Step 4d-1: 学習環境のセットアップ（kohya-ss/sd-scripts）

SD_SCRIPTS_DIR = "/content/sd-scripts"

![ ! -d $SD_SCRIPTS_DIR ] && echo -= Initial setup sd-scripts =- && git clone https://github.com/kohya-ss/sd-scripts.git $SD_SCRIPTS_DIR
%cd $SD_SCRIPTS_DIR
!pip install -q -r requirements.txt
!pip install -q toml

print("✅ sd-scripts のセットアップ完了")

In [ ]:
#@title Step 4d-2: データセットの準備（Google Drive の画像フォルダ）

import os, glob, toml

# ── 設定 ──
CHARACTER_NAME  = "my_character"  #@param {type:"string"} 出力LoRAのファイル名になります
TRIGGER_WORD    = "mychar1"  #@param {type:"string"} プロンプトで呼び出すためのユニークな単語
IMAGE_DIR       = "/content/drive/MyDrive/AnimaLoRA/my_character/images"  #@param {type:"string"} キャラクター参照画像を置いた Google Drive フォルダ
AUTO_CAPTION_IF_MISSING = True  #@param {type:"boolean"} 同名.txtが無い画像にトリガーワードのみのキャプションを自動生成
NUM_REPEATS     = 10  #@param {type:"integer"}
RESOLUTION      = 1024  #@param {type:"integer"}
MAX_TRAIN_EPOCHS = 10  #@param {type:"integer"}

assert os.path.isdir(IMAGE_DIR), f"画像フォルダが見つかりません: {IMAGE_DIR}"

image_exts = (".png", ".jpg", ".jpeg", ".webp")
images = sorted([p for p in glob.glob(f"{IMAGE_DIR}/*") if p.lower().endswith(image_exts)])
assert len(images) > 0, "画像が1枚も見つかりません"

if len(images) < 10:
    print(f"⚠️ 画像が{len(images)}枚のみです。15〜30枚程度を推奨します。")

# キャプションが無い画像にはトリガーワードのみのキャプションを補完
missing = 0
for img in images:
    txt_path = os.path.splitext(img)[0] + ".txt"
    if not os.path.exists(txt_path):
        missing += 1
        if AUTO_CAPTION_IF_MISSING:
            with open(txt_path, "w", encoding="utf-8") as f:
                f.write(TRIGGER_WORD)

print(f"画像: {len(images)}枚 / キャプション未設定だった画像: {missing}枚")
if missing and not AUTO_CAPTION_IF_MISSING:
    print("⚠️ キャプション未設定の画像があります。学習品質のため .txt キャプションの用意を推奨します。")

# ── データセット設定 (TOML) を生成 ──
dataset_config = {
    "general": {
        "resolution": RESOLUTION,
        "caption_extension": ".txt",
        "batch_size": 1,
        "shuffle_caption": False,
    },
    "datasets": [
        {
            "subsets": [
                {"image_dir": IMAGE_DIR, "num_repeats": NUM_REPEATS}
            ]
        }
    ],
}

DATASET_CONFIG_PATH = "/content/anima_dataset_config.toml"
with open(DATASET_CONFIG_PATH, "w", encoding="utf-8") as f:
    toml.dump(dataset_config, f)

print(f"✅ データセット設定を書き出しました: {DATASET_CONFIG_PATH}")

total_steps = len(images) * NUM_REPEATS * MAX_TRAIN_EPOCHS
print(f"推定ステップ数: {total_steps}")
if total_steps > 1000:
    print("⚠️ 無料版ColabのT4では1000stepsで4時間以上かかることがあります。epoch数か画像枚数を減らすことを推奨します。")

In [ ]:
#@title Step 4d-3: LoRA学習の実行

import os, subprocess

DIT_MODEL   = f"{WORKSPACE}/models/diffusion_models/anima-base-v1.0.safetensors"
QWEN3_MODEL = f"{WORKSPACE}/models/text_encoders/qwen_3_06b_base.safetensors"
VAE_MODEL   = f"{WORKSPACE}/models/vae/qwen_image_vae.safetensors"

for p in (DIT_MODEL, QWEN3_MODEL, VAE_MODEL):
    assert os.path.exists(p), f"モデルが見つかりません: {p}（Step4のAnimaモデルダウンロードを先に実行してください）"

NETWORK_DIM   = 8  #@param {type:"integer"} LoRAのランク（大きいほど表現力↑・容量↑）
LEARNING_RATE = 1e-4  #@param {type:"number"}

OUTPUT_DIR  = "/content/anima_lora_output"
OUTPUT_NAME = CHARACTER_NAME
os.makedirs(OUTPUT_DIR, exist_ok=True)

cmd = (
    "accelerate launch --num_cpu_threads_per_process 1 anima_train_network.py "
    f'--pretrained_model_name_or_path="{DIT_MODEL}" '
    f'--qwen3="{QWEN3_MODEL}" '
    f'--vae="{VAE_MODEL}" '
    f'--dataset_config="{DATASET_CONFIG_PATH}" '
    f'--output_dir="{OUTPUT_DIR}" '
    f'--output_name="{OUTPUT_NAME}" '
    "--save_model_as=safetensors "
    "--network_module=networks.lora_anima "
    f"--network_dim={NETWORK_DIM} "
    "--network_train_unet_only "
    f"--learning_rate={LEARNING_RATE} "
    "--optimizer_type=AdamW8bit "
    "--lr_scheduler=constant "
    "--timestep_sampling=sigmoid "
    "--discrete_flow_shift=1.0 "
    f"--max_train_epochs={MAX_TRAIN_EPOCHS} "
    "--mixed_precision=bf16 "
    "--gradient_checkpointing "
    "--cache_latents "
    "--cache_text_encoder_outputs"
)

result = subprocess.run(cmd, shell=True, cwd=SD_SCRIPTS_DIR)
assert result.returncode == 0, "学習に失敗しました。ログを確認してください。"
print("\n✅ 学習完了")

In [ ]:
#@title Step 4d-4: LoRAをComfyUI形式に変換して配置

import os, subprocess

trained_lora = f"{OUTPUT_DIR}/{OUTPUT_NAME}.safetensors"
assert os.path.exists(trained_lora), f"学習済みLoRAが見つかりません: {trained_lora}"

comfy_lora_dir = f"{WORKSPACE}/models/loras"
os.makedirs(comfy_lora_dir, exist_ok=True)
comfy_lora_path = f"{comfy_lora_dir}/{OUTPUT_NAME}.safetensors"

result = subprocess.run(
    ["python", f"{SD_SCRIPTS_DIR}/networks/convert_anima_lora_to_comfy.py", trained_lora, comfy_lora_path],
    cwd=SD_SCRIPTS_DIR,
)
assert result.returncode == 0, "ComfyUI形式への変換に失敗しました。"

print(f"✅ ComfyUI用LoRAを配置しました: {comfy_lora_path}")
print("")
print("【ComfyUI ワークフローでの使い方】")
print(f"  LoraLoader / LoraLoaderModelOnly で読み込み: {OUTPUT_NAME}.safetensors")
print(f"  プロンプトに必ずトリガーワードを含める     : {TRIGGER_WORD}")
print("  strength : 0.7〜1.0（キャラクターが薄い/濃すぎる場合に調整）")
print("")
print("※ Anima-Turbo LoRA と併用する場合は、キャラクターLoRA → Turbo LoRA の順に LoraLoader を重ねてください。")

## Step 5: 既存ワークフロー（PNG埋め込み）の読み込み

ComfyUI はワークフロー情報を PNG のメタデータに保存します。  
読み込みはブラウザ側で処理されるため、**ファイルを Colab にアップロードする必要はありません**。

### 読み込み方法

| 方法 | 手順 |
|------|------|
| **ドラッグ&ドロップ** | ComfyUI の画面上にローカル PC の PNG をドラッグする |
| **Load ボタン** | 右クリックメニューまたはメニューバーの `Load` からローカルの PNG を選択 |

### モデル名の不一致に注意

ワークフロー内のモデル名と実際にダウンロードされたファイル名が一致しないとエラーになります。  
下のセルで旧モデル名 → 新モデル名のシンボリックリンクを作成できます。

## Step 5: ComfyUI の起動

**cloudflared（推奨）** か **localtunnel** か **Colab iframe** の3種類から選んで実行してください。

In [ ]:
#@title 起動方法 A: cloudflared（推奨）

!wget -q -P ~ https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i ~/cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\nComfyUI の起動完了。cloudflared でトンネルを開きます...\n")
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            print("🌐 ComfyUI アクセス URL:", l[l.find("http"):], end='')

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

%cd /content/ComfyUI

# CPUで動かす場合はこちら（GPU制限解除待ちの場合）
# !python main.py --cpu --dont-print-server --listen --enable-cors-header *

# もしT4 GPUが使えるようになったら、--cpu を外して以下にしてください
# GPUで実行する場合
!python main.py --dont-print-server --listen --enable-cors-header '*'

Selecting previously unselected package cloudflared.
(Reading database ... 122363 files and directories currently installed.)
Preparing to unpack .../cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.5.0) ...
Setting up cloudflared (2026.5.0) ...
Processing triggers for man-db (2.10.2-1) ...
/content/ComfyUI
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[WARNING] WARNING: blake3 package not installed
[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-05-25 10:22:38.016
** Platform: Linux
** Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /c

In [ ]:
#@title 起動方法 B: localtunnel（cloudflared が使えない場合）

!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\nComfyUI の起動完了。localtunnel でトンネルを開きます...\n")
    endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
    print("🔑 localtunnel パスワード / エンドポイント IP:", endpoint_ip)
    p = subprocess.Popen(["lt", "--port", str(port)], stdout=subprocess.PIPE)
    for line in p.stdout:
        print(line.decode(), end='')

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

%cd /content/ComfyUI
!python main.py --dont-print-server

In [ ]:
#@title 起動方法 C: Colab iframe（WebSocket 非対応のため機能制限あり）

import threading
import time
import socket

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    from google.colab import output
    output.serve_kernel_port_as_iframe(port, height=1024)
    print("別ウィンドウで開く場合はこちら:")
    output.serve_kernel_port_as_window(port)

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

%cd /content/ComfyUI
!python main.py --dont-print-server